### **ASPEKT Paper Reviewer**


In [1]:
import json
import os
import copy
import shutil
import tempfile
from pathlib import Path
from datetime import datetime

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ── Tenta importar findpapers para download de PDF ──────────
try:
    import findpapers
    HAS_FINDPAPERS = True
except ImportError:
    HAS_FINDPAPERS = False
    print("⚠️  findpapers não encontrado. Download de PDFs desativado.")
    print("   Para ativar: pip install findpapers")

print("✅ Dependências carregadas.")

✅ Dependências carregadas.


### **Configurações**

In [2]:
# CONFIGURAÇÕES — edite conforme necessário
# ============================================================

INPUT_JSON  = r"C:\Users\vinic\OneDrive\Documentos\0300_Projetos\INCA\vfss_to_docker\data\artigos\vfss_ai_papers.json"      # Caminho para o JSON do findpapers
OUTPUT_JSON = r"C:\Users\vinic\OneDrive\Documentos\0300_Projetos\INCA\vfss_to_docker\data\artigos\selected_papers.json"      # JSON consolidado dos artigos selecionados
PDF_DIR     = r"..//data//artigos//pdfs"                      # Pasta raiz onde as subpastas por caixa serão criadas

# Instale dependências se necessário:
# pip install ipywidgets findpapers

In [3]:
# CAIXAS DO ASPEKT
# ============================================================

ASPEKT_BOXES = [
    # ── Pré-processamento / Setup ──────────────────────────
    "1. VFSS Recording & Segmentation",
    "2. Bolus Count & Subswallow Detection",

    # ── Escala de Penetração-Aspiração ────────────────────
    "3. Penetration-Aspiration Scale (PAS)",

    # ── Eventos Faríngeos ─────────────────────────────────
    "4a. Peak XT / Hyoid Tracking (velocity, position)",
    "4b. Hyoid Burst (BPM, onset/offset)",
    "4c. Laryngeal Vestibule Closure (LVC / IVA)",
    "4d. UES Opening & Maximum UES Distension",
    "4e. Maximum Pharyngeal Constriction (MPC)",
    "4f. Swallow Rest / UES Closure (UESC)",

    # ── Métricas Espaciais / Morfológicas ─────────────────
    "5a. Bolus Location & Tracking",
    "5b. UES Diameter & MPA Area",
    "5c. Pharyngeal Area at Rest",
    "5d. Normalized Residue Scale (NRS)",

    # ── Opções especiais ──────────────────────────────────
    "Múltiplas caixas",
    "Caixa não identificada",
    "Pesquisa do Som",
]

print(f"✅ {len(ASPEKT_BOXES)} caixas ASPEKT configuradas.")

✅ 16 caixas ASPEKT configuradas.


In [4]:
# CARREGAMENTO DO JSON
# ============================================================

with open(INPUT_JSON, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

papers = raw_data.get("papers", [])
print(f"✅ {len(papers)} artigos carregados de '{INPUT_JSON}'")

# Estado da sessão
selected = {}   # {index: aspekt_box}  — artigos marcados com Y

# Carrega seleções anteriores se o output já existir
if os.path.exists(OUTPUT_JSON):
    with open(OUTPUT_JSON, "r", encoding="utf-8") as f:
        prev = json.load(f)
    prev_papers = prev.get("papers", [])
    prev_titles = {p["title"]: p.get("aspekt_box") for p in prev_papers}
    for i, p in enumerate(papers):
        if p["title"] in prev_titles:
            selected[i] = prev_titles[p["title"]]
    print(f"   ↩️  {len(selected)} seleções anteriores restauradas de '{OUTPUT_JSON}'")

✅ 228 artigos carregados de 'C:\Users\vinic\OneDrive\Documentos\0300_Projetos\INCA\vfss_to_docker\data\artigos\vfss_ai_papers.json'
   ↩️  11 seleções anteriores restauradas de 'C:\Users\vinic\OneDrive\Documentos\0300_Projetos\INCA\vfss_to_docker\data\artigos\selected_papers.json'


In [5]:
# FUNÇÕES AUXILIARES
# ============================================================

def save_output():
    """Salva o JSON consolidado com os artigos selecionados."""
    out = copy.deepcopy(raw_data)
    out["papers"] = []
    out["number_of_papers"] = len(selected)
    for idx, box in selected.items():
        paper = copy.deepcopy(papers[idx])
        paper["aspekt_box"] = box
        paper["selected"] = True
        out["papers"].append(paper)
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)


def download_pdf(paper, box_name):
    """Tenta baixar o PDF do artigo usando findpapers.download."""
    if not HAS_FINDPAPERS:
        return False, "findpapers não instalado"

    # Pasta destino
    safe_box = "".join(c if c.isalnum() or c in " _-" else "_" for c in box_name).strip()
    dest_dir = Path(PDF_DIR) / safe_box
    dest_dir.mkdir(parents=True, exist_ok=True)

    # ── Copia e sanitiza os campos críticos para o downloader ──
    p = copy.deepcopy(paper)

    # urls: precisa ser uma lista (nunca None)
    urls = p.get("urls") or []
    if isinstance(urls, str):          # caso venha como string única
        urls = [urls]
    p["urls"] = list(urls)

    # doi: se existir, adiciona como URL de fallback (o downloader faz isso internamente,
    # mas garantimos que o campo esteja presente e bem-formado)
    doi = p.get("doi")
    if doi and f"http://doi.org/{doi}" not in p["urls"]:
        p["urls"].append(f"http://doi.org/{doi}")

    # keywords e databases também precisam ser listas (não None) para o from_dict não quebrar
    p["keywords"]  = list(p.get("keywords")  or [])
    p["databases"] = list(p.get("databases") or [])

    # Marca como selecionado
    p["selected"] = True

    # Monta JSON temporário
    tmp_data = copy.deepcopy(raw_data)
    tmp_data["papers"] = [p]
    tmp_data["number_of_papers"] = 1

    # Diagnóstico rápido: avisa no log se ainda não houver URLs
    if not p["urls"]:
        return False, f"Artigo sem URLs e sem DOI — impossível baixar: '{p.get('title')}'"

    tmp_path = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w", suffix=".json", delete=False, encoding="utf-8"
        ) as tf:
            json.dump(tmp_data, tf, ensure_ascii=False)
            tmp_path = tf.name

        findpapers.download(
            search_path=tmp_path,
            output_directory=str(dest_dir),
            only_selected_papers=True,
        )
        os.unlink(tmp_path)
        return True, str(dest_dir)
    except Exception as e:
        if tmp_path:
            try:
                os.unlink(tmp_path)
            except Exception:
                pass
        return False, str(e)


def fmt_authors(authors):
    if not authors:
        return "—"
    if len(authors) <= 3:
        return "; ".join(authors)
    return "; ".join(authors[:3]) + f" … (+{len(authors)-3})"


def fmt_keywords(keywords):
    if not keywords:
        return "—"
    clean = [k.lstrip("N ").strip() for k in keywords]
    return " · ".join(clean)


def fmt_urls(urls):
    if not urls:
        return "—"
    links = [f'<a href="{u}" target="_blank">{u}</a>' for u in urls]
    return "<br>".join(links)

print("✅ Funções auxiliares prontas.")

✅ Funções auxiliares prontas.


### **Aplição**

In [ ]:
# INTERFACE PRINCIPAL — layout em duas colunas
# ============================================================

state = {"idx": 0, "panel_open": False}

# ── Barra de navegação ───────────────────────────────────────
btn_prev   = widgets.Button(description="◀ Anterior",      button_style="",
                            layout=widgets.Layout(width="140px", height="38px"))
btn_next   = widgets.Button(description="Próximo ▶",       button_style="primary",
                            layout=widgets.Layout(width="140px", height="38px"))
btn_select = widgets.Button(description="★ Selecionar", button_style="success",
                            layout=widgets.Layout(width="175px", height="38px"))
counter_lbl      = widgets.Label()
selected_cnt_lbl = widgets.HTML()

nav_bar = widgets.HBox(
    [btn_prev, counter_lbl, btn_next,
     widgets.HTML("&nbsp;&nbsp;"),
     btn_select, selected_cnt_lbl],
    layout=widgets.Layout(align_items="center", gap="8px", padding="8px 0"),
)

# ── Coluna esquerda: conteúdo do artigo ─────────────────────
content_out = widgets.Output(
    layout=widgets.Layout(width="100%")
)

# ── Coluna direita: painel de classificação ASPEKT ───────────
panel_out = widgets.Output(
    layout=widgets.Layout(
        width="310px",
        min_width="310px",
        border="2px solid #e2e8f0",
        border_radius="10px",
        padding="0px",
        overflow_y="auto",
    )
)

# Painel começa oculto
panel_out.layout.display = "none"

# ── Layout de duas colunas ───────────────────────────────────
body_row = widgets.HBox(
    [content_out, panel_out],
    layout=widgets.Layout(gap="16px", align_items="flex-start", width="100%"),
)

# ── Log de ações ────────────────────────────────────────────
log_out = widgets.Output(layout=widgets.Layout(height="60px", overflow_y="auto"))

def log(msg, color="#555"):
    ts = datetime.now().strftime("%H:%M:%S")
    with log_out:
        display(HTML(f'<span style="color:{color};font-size:12px">[{ts}] {msg}</span>'))


# ── Renderização do artigo (coluna esquerda) ─────────────────
BADGE_SEL   = '<span style="background:#22c55e;color:#fff;border-radius:4px;padding:2px 10px;font-size:12px;font-weight:700">✔ SELECIONADO</span>'
BADGE_UNSEL = '<span style="background:#e5e7eb;color:#888;border-radius:4px;padding:2px 10px;font-size:12px">não selecionado</span>'

def render_paper():
    idx  = state["idx"]
    p    = papers[idx]
    is_sel = idx in selected

    counter_lbl.value = f"{idx + 1} / {len(papers)}"
    n_sel = len(selected)
    selected_cnt_lbl.value = (
        f'<span style="color:#16a34a;font-weight:600">✔ {n_sel} selecionado(s)</span>'
    )

    if is_sel:
        btn_select.description  = "✖ Desmarcar"
        btn_select.button_style = "danger"
    else:
        btn_select.description  = "★ Selecionar"
        btn_select.button_style = "success"

    if is_sel:
        box_label  = selected.get(idx, "")
        status_html = (
            BADGE_SEL +
            f' <span style="background:#dbeafe;color:#1e40af;border-radius:4px;'
            f'padding:2px 10px;font-size:12px">📂 {box_label}</span>'
        )
    else:
        status_html = BADGE_UNSEL

    pub_info = ""
    if p.get("publication"):
        pub = p["publication"]
        pub_info = pub.get("title") or ""
        if pub.get("category"):
            pub_info += f" ({pub['category']})"

    doi_html = (f'<a href="https://doi.org/{p["doi"]}" target="_blank">{p["doi"]}</a>'
                if p.get("doi") else "—")

    html = f"""
    <div style="font-family:'Segoe UI',sans-serif;padding:0 4px">
      <div style="margin-bottom:12px">{status_html}</div>
      <h2 style="margin:0 0 8px;font-size:1.2rem;color:#1e293b;line-height:1.4">
        {p.get('title') or '(sem título)'}
      </h2>
      <div style="font-size:13px;color:#64748b;margin-bottom:16px;display:flex;flex-wrap:wrap;gap:12px">
        <span>👥 {fmt_authors(p.get('authors'))}</span>
        <span>📅 {p.get('publication_date') or '—'}</span>
        <span>🏛 {pub_info or '—'}</span>
        <span>🔗 DOI: {doi_html}</span>
        <span>🗄 {', '.join(p.get('databases') or []) or '—'}</span>
      </div>
      <div style="margin-bottom:14px">
        <span style="font-size:12px;font-weight:600;color:#475569;text-transform:uppercase;letter-spacing:.05em">Palavras-chave</span><br>
        <span style="font-size:13px;color:#334155">{fmt_keywords(p.get('keywords'))}</span>
      </div>
      <div style="margin-bottom:14px">
        <span style="font-size:12px;font-weight:600;color:#475569;text-transform:uppercase;letter-spacing:.05em">Abstract</span>
        <div style="font-size:13.5px;color:#1e293b;line-height:1.65;margin-top:4px;
                    background:#f8fafc;border-left:3px solid #94a3b8;
                    padding:10px 14px;border-radius:0 6px 6px 0;
                    max-height:340px;overflow-y:auto">
          {p.get('abstract') or '<em>Abstract não disponível.</em>'}
        </div>
      </div>
      <div style="font-size:12px;color:#64748b">
        <strong>URLs:</strong> {fmt_urls(p.get('urls'))}
      </div>
    </div>
    """

    with content_out:
        clear_output(wait=True)
        display(HTML(html))


# ── Painel lateral de classificação (coluna direita) ─────────
def show_panel(idx):
    """Abre o painel lateral com as caixas ASPEKT ao lado do abstract."""
    state["panel_open"] = True
    panel_out.layout.display = ""

    current_box = selected.get(idx)

    panel_title = widgets.HTML(
        '<div style="background:#1e40af;color:#fff;padding:10px 14px;'
        'border-radius:8px 8px 0 0;font-family:sans-serif">'
        '<div style="font-size:13px;font-weight:700">📂 Caixa ASPEKT</div>'
        f'<div style="font-size:11px;opacity:.8;margin-top:2px">Artigo #{idx+1}</div>'
        '</div>'
    )

    box_select = widgets.RadioButtons(
        options=ASPEKT_BOXES,
        value=current_box if current_box in ASPEKT_BOXES else ASPEKT_BOXES[0],
        layout=widgets.Layout(width="100%"),
    )

    # Separador visual antes das opções especiais
    sep_html = widgets.HTML(
        '<hr style="border:none;border-top:1px dashed #cbd5e1;margin:4px 10px">'
        '<div style="font-size:10px;color:#94a3b8;padding:0 10px 4px">OPÇÕES ESPECIAIS</div>'
    )

    btn_ok     = widgets.Button(description="✔ Concluir", button_style="success",
                                layout=widgets.Layout(width="120px", height="34px"))
    btn_cancel = widgets.Button(description="✖ Cancelar", button_style="warning",
                                layout=widgets.Layout(width="120px", height="34px"))
    btn_row = widgets.HBox(
        [btn_ok, btn_cancel],
        layout=widgets.Layout(gap="8px", padding="10px 14px", justify_content="center"),
    )

    radio_container = widgets.VBox(
        [box_select],
        layout=widgets.Layout(padding="8px 14px"),
    )

    panel_body = widgets.VBox(
        [panel_title, radio_container, btn_row],
        layout=widgets.Layout(width="100%"),
    )

    def on_ok(_):
        chosen = box_select.value
        selected[idx] = chosen
        save_output()
        ok, info = download_pdf(papers[idx], chosen)
        if ok:
            log(f"📥 PDF salvo em: {info}", "#16a34a")
        elif HAS_FINDPAPERS:
            log(f"⚠️  Não foi possível baixar o PDF: {info}", "#b45309")
        log(f"✔ Artigo #{idx+1} → '{chosen}' | JSON salvo.", "#1d4ed8")
        close_panel()
        render_paper()

    def on_cancel(_):
        log("↩️  Classificação cancelada.", "#64748b")
        close_panel()

    btn_ok.on_click(on_ok)
    btn_cancel.on_click(on_cancel)

    with panel_out:
        clear_output(wait=True)
        display(panel_body)


def close_panel():
    state["panel_open"] = False
    panel_out.layout.display = "none"
    with panel_out:
        clear_output(wait=True)


# ── Callbacks dos botões de navegação ────────────────────────
def on_prev(_):
    if state["idx"] > 0:
        state["idx"] -= 1
        close_panel()
        render_paper()

def on_next(_):
    if state["idx"] < len(papers) - 1:
        state["idx"] += 1
        close_panel()
        render_paper()

def on_select(_):
    idx = state["idx"]
    if idx in selected:
        # Já selecionado → desmarcar diretamente
        del selected[idx]
        save_output()
        log(f"✖ Artigo #{idx+1} desmarcado.", "#dc2626")
        close_panel()
        render_paper()
    elif state["panel_open"]:
        # Painel já aberto → fechar (toggle)
        close_panel()
    else:
        # Abrir painel lateral
        show_panel(idx)
        render_paper()  # atualiza badge

btn_prev.on_click(on_prev)
btn_next.on_click(on_next)
btn_select.on_click(on_select)

# ── Captura de teclas (setas + Y) ────────────────────────────
keyboard_js = widgets.HTML("""
<script>
(function() {
  if (window._aspektListenerAttached) return;
  window._aspektListenerAttached = true;
  document.addEventListener('keydown', function(e) {
    if (['INPUT','TEXTAREA','SELECT'].includes(document.activeElement.tagName)) return;
    var btns = Array.from(document.querySelectorAll('button.widget-button'));
    if (e.key === 'ArrowRight') {
      var b = btns.find(b => b.textContent.includes('Próximo'));
      if (b) b.click();
    } else if (e.key === 'ArrowLeft') {
      var b = btns.find(b => b.textContent.includes('Anterior'));
      if (b) b.click();
    } else if (e.key === 'y' || e.key === 'Y') {
      var b = btns.find(b => b.textContent.includes('Selecionar') || b.textContent.includes('Desmarcar'));
      if (b) b.click();
    }
  });
})();
</script>
""")

# ── Montagem final ───────────────────────────────────────────
sep = widgets.HTML('<hr style="border:none;border-top:1px solid #e2e8f0;margin:4px 0">')

ui = widgets.VBox([
    keyboard_js,
    nav_bar,
    sep,
    body_row,
    sep,
    widgets.HTML('<span style="font-size:11px;color:#94a3b8">Log de ações:</span>'),
    log_out,
])

render_paper()
display(ui)

### **Análise da Sessão**

In [ ]:
# Diagnóstico geral do JSON de busca
total = len(papers)
sem_doi  = sum(1 for p in papers if not p.get("doi"))
sem_urls = sum(1 for p in papers if not p.get("urls"))
sem_ambos = sum(1 for p in papers if not p.get("doi") and not p.get("urls"))

print(f"Total de artigos : {total}")
print(f"Sem DOI          : {sem_doi}")
print(f"Sem URLs         : {sem_urls}")
print(f"Sem DOI e URLs   : {sem_ambos}")
print()

# Mostra as chaves disponíveis no primeiro artigo (para ver a estrutura real)
print("Chaves do 1º artigo:", list(papers[0].keys()))
print()

# Mostra os primeiros 3 artigos completos
for i, p in enumerate(papers[:3]):
    print(f"--- Artigo #{i} ---")
    for k, v in p.items():
        print(f"  {k}: {v}")
    print()



In [ ]:
# ============================================================
# RESUMO DA SESSÃO (execute quando quiser ver o status)
# ============================================================

print(f"Total de artigos: {len(papers)}")
print(f"Artigos selecionados: {len(selected)}")
print()
if selected:
    from collections import Counter
    counts = Counter(selected.values())
    print("Distribuição por caixa ASPEKT:")
    for box, n in sorted(counts.items()):
        print(f"  {box}: {n}")
else:
    print("Nenhum artigo selecionado ainda.")